In [ ]:
import itertools
import math
from math import pi
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.interpolate
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.magnus6 import sesolve_magnusgl6
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.simulation.signals import planck_taper_signal
from fluxoniumcr.utils import load_arguments

In [ ]:
parent_path = DATA_DIR/"control_qubit_collisions/EJ=4.00,EC=1.20,EL=0.40"
argm = load_arguments(parent_path)

amp_from_deltap_dataset = xr.load_dataset(
    DATA_DIR/"charge_operator/EJ=4.00,EC=1.20,EL=0.40/amplitude_from_deltap.hdf5"
)

In [ ]:
fx = Fluxonium(
    EJ=argm.EJ,
    EC=argm.EC,
    EL=argm.EL,
    dim=argm.levels,
    cutoff=argm.cutoff,
    flux=0.5,
)

E = fx.eigenvalues
H0_op = np.diag(E)
n_op = fx.get_operator('charge')

fx_frequency = (E[1] - E[0])
Ω0 = fx_frequency/abs(n_op[0, 1])


collision_window_threshold = argm.collision_window_threshold

In [ ]:
def calculate_transition_probability(
        frequency: float,
        amplitude: float,
        ramp_duration: float,
        dt: float = 0.01,
):
    carrier_period = 2*pi/frequency
    signal = planck_taper_signal(
        amplitude=amplitude,
        total_duration=2*ramp_duration + carrier_period,
        ramp_duration=ramp_duration,
        phase=-pi/2,
        t0=-ramp_duration,
        carrier_freq=frequency,
    )
    
    result_ramp = sesolve_magnusgl6(
        [
            H0_op,
            [n_op, signal],
        ],
        np.identity(H0_op.shape[0]),
        tlist=np.linspace(-ramp_duration, 0, math.ceil(ramp_duration/dt)),
    )
    U_ramp = result_ramp.y[-1]

    result_period = sesolve_magnusgl6(
        [
            H0_op,
            [n_op, signal],
        ],
        np.identity(H0_op.shape[0]),
        tlist=np.linspace(0, carrier_period, 256),
    )
    U_period = result_period.y[-1]

    evals, evecs = np.linalg.eig(U_period)
    rho1 = np.abs(evecs.T.conj() @ U_ramp)**2
    rho1 = np.expand_dims(rho1.T, axis=2)
    rho2 = (U_ramp.T @ evecs) @ (rho1 * np.expand_dims((U_ramp.T @ evecs).T.conj(),axis=0))
    transition_probs = np.diagonal(rho2, axis1=-2, axis2=-1).real.T
    
    return transition_probs

In [ ]:
amp_from_deltap = xr.DataArray(
    None,
    coords=[amp_from_deltap_dataset.deltap],
)
for deltap in amp_from_deltap_dataset.deltap.data:
    ds = amp_from_deltap_dataset.sel(deltap=deltap)
    y = ds.amplitude_spline.dropna('frequency')
    amp_from_deltap.loc[dict(deltap=deltap)] = (
        lambda x, y=y:
            np.interp(x, y.frequency, y, left=np.nan, right=np.nan)
    )

In [ ]:
dataset = xr.Dataset(
    attrs=dict(
        EJ=argm.EJ,
        EC=argm.EC,
        EL=argm.EL,
        ramp_shape=argm.ramp_shape,
    )
)

frequency_coord = xr.DataArray(
    argm.frequencies,
    dims=['frequency'],
    attrs=dict(
        long_name="Drive frequency",
        units="Grad/s",
    )
)
deltap_coord = xr.DataArray(
    argm.delta_polarizations,
    dims=['deltap'],
    attrs=dict(
        long_name="Conditional polarization magnitude",
    )
)
ramp_duration_coord = xr.DataArray(
    argm.ramp_durations,
    dims=['ramp_duration'],
    attrs=dict(
        long_name="Ramp duration",
        units="ns",
    )
)

level_data = np.arange(argm.levels, dtype=np.uint8)
bra_coord = xr.DataArray(level_data, dims=['bra'])
ket_coord = xr.DataArray(level_data, dims=['ket'])

dataset['amplitude'] = xr.DataArray(
    np.nan,
    coords=[frequency_coord, deltap_coord],
    attrs=dict(
        long_name="Drive amplitude",
        units="Grad/s",
    )
)
dataset['amplitude'].encoding['scale_factor'] = Ω0

dataset['transition_probability'] = xr.DataArray(
    np.float32('nan'),
    coords=[
        frequency_coord,
        deltap_coord,
        ramp_duration_coord,
        bra_coord,
        ket_coord,
    ],
    attrs=dict(
        long_name="Transition probability"
    )
)

In [ ]:
for deltap in dataset.deltap.data:
    ds = dataset.loc[dict(deltap=deltap)]
    ds.amplitude[:] = amp_from_deltap.sel(deltap=deltap).item()(ds.frequency.data)

In [ ]:
for (
        frequency,
        deltap,
        ramp_duration
) in tqdm.contrib.itertools.product(
        dataset.frequency.data,
        dataset.deltap.data,
        dataset.ramp_duration.data,
):
    ds = dataset.loc[dict(
        frequency=frequency,
        deltap=deltap,
        ramp_duration=ramp_duration,
    )]
    amplitude = ds.amplitude.item()
    if np.isnan(amplitude): continue
    if not np.isnan(ds.transition_probability.data.ravel()[0]): continue
    probs = calculate_transition_probability(
        frequency=frequency,
        amplitude=amplitude,
        ramp_duration=ramp_duration
    )
    ds['transition_probability'][()] = probs

In [ ]:
dataset.to_netcdf(parent_path/f"{argm.ramp_shape}.hdf5")

In [ ]:
dataset = xr.load_dataset(parent_path/f"{argm.ramp_shape}.hdf5")

In [ ]:
transitions_dict = {}

for deltap, ramp_duration in itertools.product(
        dataset.deltap.data,
        dataset.ramp_duration.data,
):
    this_dict = {}
    
    for ket in [0, 1]:
        ds = dataset.sel(
            ramp_duration=ramp_duration,
            deltap=deltap,
            ket=ket,
        )
        for bra in ds.bra.data:
            if bra == ket: continue
            prob = ds.transition_probability.sel(bra=bra)
            peak_indices, peak_properties = scipy.signal.find_peaks(
                prob.data,
                height=collision_window_threshold,
            )

            (
                peak_prominences,
                left_base_indices,
                right_base_indices
            ) = scipy.signal.peak_prominences(
                np.log10(prob.data),
                peak_indices,
            )

            # Keep only peaks with prominences at least 10x the contour
            mask = peak_prominences > 1

            peak_indices = peak_indices[mask]
            peak_freqs = ds.frequency.data[peak_indices]
            peak_heights = peak_properties['peak_heights'][mask]
            left_bases = ds.frequency.data[left_base_indices[mask]]
            right_bases = ds.frequency.data[right_base_indices[mask]]

            for center_freq, height, left_base, right_base in zip(
                    peak_freqs,
                    peak_heights,
                    left_bases,
                    right_bases,
            ):
                bare_freq = H0_op[bra,bra] - H0_op[ket,ket]
                if (int(bra) - int(ket)) % 2 == 0:
                    harmonic = round(bare_freq/center_freq/2)*2
                else:
                    harmonic = round((bare_freq/center_freq - 1)/2)*2 + 1
                
                key = (harmonic, int(bra), int(ket))
                
                if key in this_dict and this_dict[key][0] >= height:
                    # Another larger peak already exists.
                    continue
                    
                this_dict[key] = (
                    height,
                    center_freq,
                    left_base,
                    right_base,
                )
                print(
                    f"({harmonic}, {bra}, {ket})\t",
                    "{:.3g}\t\t {:+.3g}".format(
                        center_freq/(2*pi),
                        (center_freq - bare_freq/harmonic)*1e3/(2*pi)
                    ),
                )
    
    if (1, 1, 0) not in this_dict:
        # Manually add this peak in since find_peaks won't find a "half" peak.
        ds = dataset.sel(
            ramp_duration=ramp_duration,
            deltap=deltap,
            bra=1,
            ket=0,
        ).dropna('frequency')
        height = ds.transition_probability.data[0]
        fc = ds.transition_probability.frequency.data[0]
        this_dict[1, 1, 0] = (
            height,
            fc,
            fc,
            fc + 0.5 * 2*pi,  # Set right base to a fixed +500 MHz.
        )
        
    transitions_dict[deltap, ramp_duration] = this_dict

In [ ]:
collision_windows_dataset = xr.Dataset()

all_transition_labels = sorted(
    set.union(*(set(d.keys()) for d in transitions_dict.values()))
    - {(-1, 0, 1)}
)
transition_index_coord = xr.DataArray(
    np.arange(len(all_transition_labels)),
    dims=['transition_index']
)

for key in ("fc", "fl", "fr"):
    collision_windows_dataset[key] = xr.DataArray(
        np.nan,
        coords=[
            dataset.deltap,
            dataset.ramp_duration,
            transition_index_coord,
        ]
    )
    
for i, name in enumerate(("harmonic", "bra", "ket")):
    collision_windows_dataset[name] = xr.DataArray(
        [k[i] for k in all_transition_labels],
        coords=[
            transition_index_coord,
        ]
    )

In [ ]:
min_leakage_prob = 1e-4

for deltap, ramp_duration, transition_index in tqdm.contrib.itertools.product(
        collision_windows_dataset.deltap.data,
        collision_windows_dataset.ramp_duration.data,
        collision_windows_dataset.transition_index.data,
):
    ds = collision_windows_dataset.loc[dict(
        deltap=deltap,
        ramp_duration=ramp_duration,
        transition_index=transition_index,
    )]
    harmonic = ds.harmonic.item()
    bra = ds.bra.item()
    ket = ds.ket.item()
    
    try:
        height, fc, fa, fb = transitions_dict[deltap, ramp_duration][harmonic, bra, ket]
    except KeyError:
        continue
        
    
    def fun(freq, thresh=0.0):
        probs = calculate_transition_probability(
            frequency=freq,
            amplitude=amp_from_deltap.sel(deltap=deltap).item()(freq),
            ramp_duration=ramp_duration
        )
        return probs[bra, ket] - thresh
    
    if fun(fc) < collision_window_threshold:
        continue
    
    ds.fc[()] = fc
    
    if (harmonic, bra, ket) != (1, 1, 0):
        left_and_right = [0, 1]
    else:
        # Find fr only, fl is undefined since we can't drive below qubit frequency by much.
        left_and_right = [1]
    for i in left_and_right:
        result = scipy.optimize.root_scalar(
            fun,
            args=(min_leakage_prob,),
            bracket=(fc, (fa, fb)[i])
        )
        ds[('fl', 'fr')[i]][()] = result.root

In [ ]:
ds = collision_windows_dataset.sel(ramp_duration=50, transition_index=1)
print(ds.harmonic.item(), ds.bra.item(), ds.ket.item())
ds.fc.plot()
ds.fl.plot()
ds.fr.plot()
plt.axhline((E[ds.bra.item()] - E[ds.ket.item()])/ds.harmonic.item(), c='red')

# plt.axhline(1.5*(2*pi), c='blue')

## Collision windows analysis

In [ ]:
from contextlib import redirect_stdout

ramp_duration = 50
deltap = 1.0

ds = collision_windows_dataset.sel(
    ramp_duration=ramp_duration,
    deltap=deltap,
)

mask = (ds.fr - ds.fl) >= 1e-3 * 2*pi
ds_masked = ds.sel(transition_index=mask)

collisions_file_name = parent_path/f"collision_windows_ramp_shape={dataset.ramp_shape}_{deltap=}_{ramp_duration=}.txt"


with open(collisions_file_name, 'w') as fp:
    with redirect_stdout(fp):
        print(f"{deltap=}, {ramp_duration=}, ")
        print("[harmonic, final, initial]")
        print("\t\t[GHz]\t[MHz]\t[MHz]")
        for tidx, dst in ds.groupby('transition_index'):
            if (dst.fr - dst.fl) < 1e-3 * 2*pi: continue
            harmonic = dst.harmonic.item()
            bra = dst.bra.item()
            ket = dst.ket.item()
            fc = dst.fc.item()
            fr = dst.fr.item()
            fl = dst.fl.item()
            fbare = (E[bra] - E[ket])/harmonic

            if np.isnan(fc): continue

            # Ignore collisions above 1.5 GHz
            if fl > 1.5 * 2*pi: continue

            print(
                "{:2d}, {}, {}"
                .format(harmonic, bra, ket)
            )
            print(
                "\tBare:\t{:.3f}\t{:+.1f}\t{:+.1f}"
                .format(fbare/(2*pi), 1e3*(fr-fbare)/(2*pi), 1e3*(fl-fbare)/(2*pi))
            )
            print(
                "\tDress:\t{:.3f}\t{:+.1f}\t{:+.1f}"
                .format(fc/(2*pi), 1e3*(fr-fc)/(2*pi), 1e3*(fl-fc)/(2*pi))
            )